[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch02-ml-systems/03-bon-moi-truong-theo-bac-do-lon.ipynb)

# Bốn môi trường theo bậc độ lớn: bộ nhớ, năng lượng, nhiệt và tiền

Chương nói khác biệt giữa các môi trường triển khai là khác biệt về chất, không chỉ về lượng. Khoảng
cách giữa chúng đo bằng bậc độ lớn, và chính khoảng cách đó buộc mỗi môi trường phải có cách làm kỹ thuật
riêng. Sổ tay
này đặt các con số của chương lên cùng một thang log, rồi hỏi từng câu mà mỗi môi trường đặt ra: mô hình
có vừa bộ nhớ không, một lần suy luận (inference) tốn bao nhiêu năng lượng, pin và nhiệt cho chạy bao lâu, và cả hệ
thống tốn bao nhiêu tiền.

**Bạn sẽ làm:**
1. đặt bộ nhớ và công suất của bốn thiết bị đại diện lên thang log, và đếm số bậc độ lớn;
2. lập bảng xem trọng số của từng mô hình đại diện (lighthouse model) có vừa từng thiết bị không, ở FP32
   và ở INT8;
3. kiểm lại bảng năng lượng cho mỗi lần suy luận của chương, rồi đổi nó ra công suất trung bình;
4. tính thuế pin, tường nhiệt, và kho năng lượng của một cảm biến thu hoạch năng lượng (energy
   harvesting);
5. làm lại phép tính tổng chi phí sở hữu (TCO) cloud và edge, và chọn bộ tăng tốc (accelerator) cho 500 cửa hàng;
6. tính một thiết bị mỗi môi trường cần bao lâu để huấn luyện GPT-3.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Chín bậc độ lớn

Theo bảng phổ triển khai của chương, công suất đi từ mê-ga-oát của trung tâm dữ liệu tới mi-li-oát của
vi điều khiển, còn bộ nhớ đi từ terabyte tới kilobyte. Bảng phần cứng của chương gắn mỗi môi trường với
một thiết bị cụ thể. Sổ tay dùng đơn vị thập phân (1 KB = 1,000 byte), như chính các phép tính của chương.

**Dự đoán:** bộ nhớ của một TPU v4 Pod gấp bao nhiêu lần bộ nhớ của một vi điều khiển ESP32-S3? Một
nghìn, một triệu, hay hơn?

In [ ]:
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # cloud, edge, mobile, TinyML; thứ tự cố định
KB, MB, GB, TB = 1e3, 1e6, 1e9, 1e12


def co_watt(w):
    """Viết một công suất theo đơn vị dễ đọc."""
    return f"{w / 1e6:,.3g} MW" if w >= 1e6 else f"{w:,.3g} W"


def co_byte(b):
    """Viết một dung lượng theo đơn vị thập phân dễ đọc."""
    for chia, don_vi in ((TB, "TB"), (GB, "GB"), (MB, "MB"), (KB, "KB")):
        if b >= chia:
            return f"{b / chia:,.4g} {don_vi}"
    return f"{b:,.0f} B"


# bảng phần cứng của chương: bộ nhớ (byte), công suất (W, thấp và cao), giá (đô la)
THIET_BI = {
    "TPU v4 Pod (cloud)": dict(
        bo_nho=sach(128, trich="128 TB HBM2") * TB,
        cong_suat=(sach(3, trich="~3 MW", tol=0.5) * 1e6,) * 2,
        gia=None),
    "DGX Spark (edge)": dict(
        bo_nho=sach(128, trich="128 GB LPDDR5x") * GB,
        cong_suat=(sach(200, trich="~200 W", tol=0.5),) * 2,
        gia=(sach(3000, nguon="128 GB, 4 TB, ~200 W, $3,000–5,000", tol=0.5),
             sach(5000, nguon="128 GB, 4 TB, ~200 W, $3,000–5,000", tol=0.5))),
    "điện thoại flagship (mobile)": dict(
        bo_nho=sach(8, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+") * GB,
        bo_nho_cao=sach(16, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+") * GB,
        cong_suat=(sach(3, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+"),
                   sach(5, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+")),
        gia=(sach(999, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+"),) * 2),
    "ESP32-S3 (TinyML)": dict(
        bo_nho=sach(512, trich="512 KB SRAM") * KB,
        cong_suat=(sach(0.05, trich="0.05 W–1.2 W active board power"),
                   sach(1.2, trich="0.05 W–1.2 W active board power")),
        gia=(sach(10, nguon="240 MHz, 512 KB SRAM, 4 MB flash, $10"),) * 2),
}

print(f"{'thiết bị':<30} {'bộ nhớ':>10} {'công suất':>16} {'log10 bộ nhớ':>13} {'log10 công suất':>16}")
for ten, tb in THIET_BI.items():
    thap, cao = tb["cong_suat"]
    cs = co_watt(thap) if thap == cao else f"{thap:,.3g}–{cao:,.3g} W"
    lg = f"{math.log10(thap):.1f}" if thap == cao else f"{math.log10(thap):.1f}–{math.log10(cao):.1f}"
    print(f"{ten:<30} {co_byte(tb['bo_nho']):>10} {cs:>16} {math.log10(tb['bo_nho']):>13.1f} {lg:>16}")

theo_sach("bậc độ lớn công suất, từ MW tới mW", math.log10(1e6 / 1e-3), sach=9, nguon="9 bậc độ lớn")
pod, esp = THIET_BI["TPU v4 Pod (cloud)"], THIET_BI["ESP32-S3 (TinyML)"]
theo_sach("bộ nhớ TPU v4 Pod ÷ bộ nhớ ESP32-S3 (lần)", pod["bo_nho"] / esp["bo_nho"], sach=250e6, nguon="250 triệu lần")
print(f"(đếm theo 1 KB = 1,024 byte thì là {128 * 2**40 / (512 * 2**10) / 1e6:.0f} triệu lần)")
print(f"riêng hai thiết bị này cách nhau {math.log10(pod['cong_suat'][0] / esp['cong_suat'][0]):.1f} bậc về công suất")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
ten_tb = list(THIET_BI)
for i, (ten, tb) in enumerate(THIET_BI.items()):
    bn = (tb["bo_nho"], tb.get("bo_nho_cao", tb["bo_nho"]))
    for ax, (thap, cao), don_vi in ((a1, bn, "B"), (a2, tb["cong_suat"], "W")):
        ax.plot([thap, cao], [i, i], color=MAU[i], lw=6, solid_capstyle="round")
        ax.plot([thap, cao], [i, i], "o", color=MAU[i], ms=8, mec="white", mew=1.5)
for ax, ten_truc, xlim in ((a1, "bộ nhớ (byte, thang log)", (1e4, 1e16)), (a2, "công suất (W, thang log)", (1e-3, 1e8))):
    ax.set_xscale("log")
    ax.set_xlim(*xlim)
    ax.set_xlabel(ten_truc)
    ax.grid(axis="x", alpha=0.3)
a1.set_yticks(range(len(ten_tb)), ten_tb)
a1.invert_yaxis()
for i, tb in enumerate(THIET_BI.values()):
    bn_txt = co_byte(tb["bo_nho"]) + (f"–{co_byte(tb['bo_nho_cao'])}" if "bo_nho_cao" in tb else "")
    a1.annotate(bn_txt, (tb.get("bo_nho_cao", tb["bo_nho"]), i), xytext=(10, -3), textcoords="offset points", fontsize=8)
    thap, cao = tb["cong_suat"]
    cs_txt = co_watt(thap) if thap == cao else f"{thap:,.3g}–{cao:,.3g} W"
    a2.annotate(cs_txt, (cao, i), xytext=(10, -3), textcoords="offset points", fontsize=8)
a1.set_title("Bộ nhớ: 250 triệu lần")
a2.set_title("Công suất: gần tám bậc giữa bốn thiết bị này")
plt.tight_layout()
plt.show()

Trên thang tuyến tính, ESP32-S3 không hiện nổi một điểm ảnh bên cạnh TPU v4 Pod. Thang log là cách duy
nhất để thấy cả bốn. Các khoảng trống không đều: về bộ nhớ, DGX Spark chỉ hơn điện thoại khoảng một bậc,
còn điện thoại hơn ESP32-S3 hơn bốn bậc. Bảng phổ triển khai của chương xếp bốn môi trường từ MW tới mW,
tức chín bậc; bốn thiết bị cụ thể ở đây chiếm một khoảng hẹp hơn bên trong khoảng đó.

## 2. Mô hình nào vừa thiết bị nào

Điều kiện đầu tiên là trọng số phải nằm được trong bộ nhớ: số tham số nhân số byte mỗi tham số. FP32
dùng 4 byte, FP16 dùng 2, INT8 dùng 1. Đây chỉ là điều kiện cần, vì giá trị kích hoạt (activation),
bộ đệm tiền xử lý và phần mềm lúc chạy còn chiếm thêm.

**Dự đoán:** lượng tử hóa (quantization) từ FP32 xuống INT8 làm trọng số nhỏ đi 4 lần. Có mô hình nào nhờ vậy mà
chuyển được từ cloud xuống điện thoại, hay từ điện thoại xuống vi điều khiển không?

In [ ]:
BYTE = {"FP32": 4, "FP16": 2, "INT8": 1}
MO_HINH = {   # số tham số
    "phát hiện từ khoá": sach(200e3, nguon="200K tham số, 800 KB, 20 MFLOP"),
    "ResNet-50": sach(25.6, nguon="8.2 GFLOP, 25.6 triệu tham số, 102.4 MB") * 1e6,
    "GPT-2": sach(1.5e9, nguon="1.5 tỉ / 7–70 tỉ"),
    "Llama 7B": sach(7, nguon="1.5 tỉ / 7–70 tỉ") * 1e9,
    "Llama 70B": sach(70e9, nguon="1.5 tỉ / 7–70 tỉ"),
}
theo_sach("phát hiện từ khoá ở FP32 (KB)", MO_HINH["phát hiện từ khoá"] * 4 / KB, sach=800, nguon="200K tham số, 800 KB, 20 MFLOP")
theo_sach("ResNet-50 ở FP32 (MB)", MO_HINH["ResNet-50"] * 4 / MB, sach=102.4, nguon="8.2 GFLOP, 25.6 triệu tham số, 102.4 MB")

EDGE_THAP = sach(512, nguon="512 MB RAM, 1 GFLOP/s") * MB
DICH = {   # bộ nhớ của từng thiết bị, byte
    "ESP32-S3\nSRAM 512 KB": esp["bo_nho"],
    "edge cấp thấp\n512 MB": EDGE_THAP,
    "điện thoại\n8 GB": THIET_BI["điện thoại flagship (mobile)"]["bo_nho"],
    "DGX Spark\n128 GB": THIET_BI["DGX Spark (edge)"]["bo_nho"],
    "TPU v4 Pod\n128 TB": pod["bo_nho"],
}


def du_dia(tham_so, byte_moi_ts, bo_nho):
    """Bộ nhớ chia cho trọng số: ≥ 1 là trọng số vừa, chưa tính activation hay phần mềm lúc chạy."""
    return bo_nho / (tham_so * byte_moi_ts)


def boi(r, mathtext=False):
    """Một bội số cho dễ đọc: 2.6, 0.034, hoặc 10^8 khi quá lớn hay quá nhỏ."""
    if 0.01 <= r < 10:
        return f"{r:.2g}"
    if 10 <= r < 1000:
        return f"{r:.0f}"
    k = round(math.log10(r))   # bậc độ lớn gần nhất
    return f"$10^{{{k}}}$" if mathtext else f"10^{k}"


ten_dich = [d.split("\n")[0] for d in DICH]
for do_cx in ("FP32", "INT8"):
    print(f"\n{do_cx:<17} " + " | ".join(f"{d:>13}" for d in ten_dich))
    for ten, ts in MO_HINH.items():
        o = [du_dia(ts, BYTE[do_cx], bn) for bn in DICH.values()]
        print(f"{ten:<17} " + " | ".join(f"{'vừa' if r >= 1 else 'thiếu':>5} ×{boi(r):<6}" for r in o))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list("thieu_vua", [MAU[1], "#d9d9d9", MAU[0]])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, do_cx in zip(axs, ("FP32", "INT8")):
    M = np.array([[math.log10(du_dia(ts, BYTE[do_cx], bn)) for bn in DICH.values()] for ts in MO_HINH.values()])
    ax.imshow(np.clip(M, -4, 4), cmap=cmap, vmin=-4, vmax=4, aspect="auto")
    for r in range(M.shape[0]):
        for c in range(M.shape[1]):
            chu = ("vừa" if M[r, c] >= 0 else "thiếu") + "\n×" + boi(10 ** M[r, c], mathtext=True)
            ax.text(c, r, chu, ha="center", va="center", fontsize=7.5,
                    color="white" if abs(M[r, c]) > 2.5 else "0.1")
    ax.set_xticks(range(len(DICH)), list(DICH), fontsize=8)
    ax.set_title(f"trọng số ở {do_cx}")
    ax.grid(False)
axs[0].set_yticks(range(len(MO_HINH)), list(MO_HINH))
fig.suptitle("Bộ nhớ chia cho trọng số (xanh: vừa, cam: thiếu; chưa tính activation)", fontsize=10)
plt.tight_layout()
plt.show()

Cột ESP32-S3 dùng SRAM 512 KB, vì chương nói ngân sách dưới mili-oát buộc toàn bộ trạng thái suy luận
nằm trong SRAM trên chip. Thẻ điểm chuông cửa ở dưới thì tính theo bộ nhớ 4 MB mà chương dùng.

INT8 đổi được vài ô sát ranh giới: phát hiện từ khoá vào được SRAM của ESP32-S3, Llama 7B vừa khít bộ
nhớ 8 GB của điện thoại (chỉ phần trọng số, không còn chỗ cho gì khác), và Llama 70B vừa DGX Spark. Nhưng
lượng tử hóa chỉ dời ranh giới 4 lần, trong khi từ SRAM của vi điều khiển lên đích edge cấp thấp đã là
một nghìn lần, và từ một DGX Spark lên cả một TPU v4 Pod cũng vậy.

Chương có hai cạm bẫy về chính chuyện này. Một mô hình ngôn ngữ tỉ tham số không thể nằm trên vi điều
khiển 256 KB RAM, dù có giảm kích thước mô hình thế nào. Và khi mô hình phải gói trong 64 KB, lượng tử
hóa từ FP32 xuống INT8 là không đủ, buộc phải thiết kế lại cả kiến trúc.

In [ ]:
RAM_256 = sach(256, trich="microcontroller with 256 KB RAM") * KB
GOI_64 = sach(64, trich="must fit in 64 KB") * KB
mot_ti = 1e9   # "billion-parameter": một tỉ tham số
print(f"mô hình 1 tỉ tham số, giả định 4 bit mỗi tham số: {mot_ti * 0.5 / MB:,.0f} MB, "
      f"gấp {mot_ti * 0.5 / RAM_256:,.0f} lần 256 KB")
kws_int8 = MO_HINH["phát hiện từ khoá"] * BYTE["INT8"]
print(f"phát hiện từ khoá ở INT8: {kws_int8 / KB:.0f} KB, gấp {kws_int8 / GOI_64:.1f} lần 64 KB; "
      f"64 KB ở INT8 chỉ chứa {GOI_64 / BYTE['INT8']:,.0f} tham số")
assert kws_int8 > GOI_64   # INT8 thôi thì không đủ: phải bớt tham số, tức đổi kiến trúc

# Khoảng cách bộ nhớ mobile ↔ TinyML mà chương gọi là 10,000 lần
MCU = (sach(256, trich="256 KB to 1 MB of memory vs. mobile devices with 8–16 GB", ) * KB,
       sach(1, trich="256 KB to 1 MB of memory vs. mobile devices with 8–16 GB") * MB)
DT = (sach(8, trich="256 KB to 1 MB of memory vs. mobile devices with 8–16 GB") * GB,
      sach(16, trich="256 KB to 1 MB of memory vs. mobile devices with 8–16 GB") * GB)
CHENH = sach(10_000, trich=r"a 10,000\(\times\) difference")
thap, cao = DT[0] / MCU[1], DT[1] / MCU[0]
print(f"mobile ÷ TinyML: từ {thap:,.0f} đến {cao:,.0f} lần, tức 10^{math.log10(thap):.1f} đến 10^{math.log10(cao):.1f}; "
      f"sách nói {CHENH:,} lần, một bậc độ lớn chứ không phải một con số chính xác")

# TinyML so với cloud: 10⁶ đến 10⁹ lần, tuỳ vi điều khiển có mê-ga-bai hay ki-lô-bai
assert sach(1e6, nguon="10⁶–10⁹ lần") <= pod["bo_nho"] / esp["bo_nho"] <= sach(1e9, nguon="10⁶–10⁹ lần")
print(f"TPU v4 Pod ÷ ESP32-S3: 10^{math.log10(pod['bo_nho'] / esp['bo_nho']):.1f}, nằm trong khoảng 10⁶ đến 10⁹")

### Thẻ điểm của chuông cửa, và đích edge cấp thấp

Chương kiểm tính khả thi theo thứ bậc: bộ nhớ trước, độ trễ (latency) sau. Ở chuông cửa thông minh, mô hình chiếm
một phần nhỏ bộ nhớ 4 MB của ESP32-S3, nên cấp 1 đạt. Nhưng độ trễ 101 ms gấp đôi mục tiêu 50 ms, nên cấp 2
trượt. Sổ tay **giả định** mô hình của chuông cửa chính là mô hình phát hiện từ khoá 800 KB, vì chương nói
chuông cửa dùng đúng mẫu đó.

In [ ]:
BN_CHUONG = sach(4, nguon="4 MB, 101 ms, 50 ms") * MB
TRE_CHUONG = sach(101, nguon="4 MB, 101 ms, 50 ms")
MUC_TIEU = sach(50, nguon="4 MB, 101 ms, 50 ms")
for cap, ten, can, co in ((1, "bộ nhớ", MO_HINH["phát hiện từ khoá"] * 4, BN_CHUONG), (2, "độ trễ", TRE_CHUONG, MUC_TIEU)):
    print(f"cấp {cap} ({ten}): cần ÷ có = {can / co * 100:5.1f} % → {'ĐẠT' if can <= co else 'TRƯỢT'}")

# đích edge cấp thấp: ResNet-50 vừa bộ nhớ, nhưng một lần suy luận mất bao lâu ở 1 GFLOP/s?
R50_FLOP = sach(8.2, nguon="8.2 GFLOP, 25.6 triệu tham số, 102.4 MB") * 1e9
TOC_THAP = sach(1, nguon="512 MB RAM, 1 GFLOP/s") * 1e9
MOBILENET = sach(13.7, nguon="13.7×")
KHUNG_AR = sach(16.7, nguon="16.7 ms")
print(f"\nResNet-50 trên đích edge cấp thấp: trọng số FP32 chiếm {MO_HINH['ResNet-50'] * 4 / EDGE_THAP * 100:.0f} % "
      f"bộ nhớ, nhưng một lần suy luận mất {R50_FLOP / TOC_THAP:.1f} s ở đúng thông lượng đỉnh")
print(f"bớt {MOBILENET}× tính toán như MobileNetV2: {R50_FLOP / MOBILENET / TOC_THAP * 1e3:.0f} ms, "
      f"vẫn gấp {R50_FLOP / MOBILENET / TOC_THAP * 1e3 / KHUNG_AR:.0f} lần một khung AR {KHUNG_AR} ms")

Chương gọi đây là cạm bẫy đem nguyên mô hình huấn luyện trên máy bàn xuống thiết bị: trọng số FP32 của
ResNet-50 chỉ khoảng 102.4 MB, nhưng cả quy trình trên máy bàn có thể cần hàng gigabyte, và 8.2 GFLOP mỗi
lần suy luận thì bộ xử lý 1 GFLOP/s không chạy nổi. Qua được cấp bộ nhớ chưa có nghĩa là qua được cấp
thời gian.

## 3. Năng lượng cho mỗi lần suy luận

Bảng năng lượng của chương đổi năng lượng cho mỗi lần suy luận ra số lần suy luận mà một viên pin điện
thoại 3.7 V, 3000 mAh chạy được. Chương ghi rõ đây là ước lượng phân tích cho thiết bị, không phải phép đo
toàn hệ thống. Pin đó chứa $3.7 \times 3 \times 3600$ jun.

**Dự đoán:** một truy vấn GPT-4 và một lần phát hiện từ khoá cách nhau bao nhiêu bậc độ lớn?

In [ ]:
VON = sach(3.7, trich="3.7 V, 3000 mAh")
MAH = sach(3000, trich="3.7 V, 3000 mAh")
PIN_J = VON * MAH / 1000 * 3600
print(f"pin: {VON} V × {MAH} mAh = {PIN_J / 3600:.1f} Wh = {PIN_J:,.0f} J")

NANG_LUONG = {   # jun mỗi lần suy luận
    "GPT-4, một truy vấn (cloud)": sach(1, nguon="~1 kJ / 40 lượt", tol=0.5) * 1e3,
    "ResNet-50, máy chủ (cloud)": sach(75.5, nguon="75.5 mJ / 529,505") / 1e3,
    "ResNet-50, Jetson (edge)": sach(15.9, nguon="15.9 mJ / 2,511,628") / 1e3,
    "MobileNet, NPU (mobile)": sach(2.60, nguon="2.60 mJ / 15,393,505") / 1e3,
    "phát hiện từ khoá (TinyML)": sach(10, nguon="10 µJ") / 1e6,
}
theo_sach("GPT-4: lượt mỗi pin", PIN_J / NANG_LUONG["GPT-4, một truy vấn (cloud)"], sach=40, nguon="~1 kJ / 40 lượt", tol=0.5)
theo_sach("phát hiện từ khoá: lượt mỗi pin", PIN_J / NANG_LUONG["phát hiện từ khoá (TinyML)"], sach=3996e6, nguon="3996 triệu lượt")

# Ba dòng giữa: số lượt chương in ra chính xác tới từng lượt, còn năng lượng chỉ có ba chữ số.
# Nên kiểm theo chiều ngược: năng lượng = pin ÷ số lượt của sách, rồi so với năng lượng sách in.
LUOT = {"ResNet-50, máy chủ (cloud)": (sach(529_505, nguon="75.5 mJ / 529,505"), 75.5, "75.5 mJ / 529,505"),
        "ResNet-50, Jetson (edge)": (sach(2_511_628, nguon="15.9 mJ / 2,511,628"), 15.9, "15.9 mJ / 2,511,628"),
        "MobileNet, NPU (mobile)": (sach(15_393_505, nguon="2.60 mJ / 15,393,505"), 2.60, "2.60 mJ / 15,393,505")}
theo_sach("ResNet-50 máy chủ: pin ÷ số lượt (mJ)", PIN_J / LUOT["ResNet-50, máy chủ (cloud)"][0] * 1e3, sach=75.5, nguon="75.5 mJ / 529,505")
theo_sach("ResNet-50 Jetson: pin ÷ số lượt (mJ)", PIN_J / LUOT["ResNet-50, Jetson (edge)"][0] * 1e3, sach=15.9, nguon="15.9 mJ / 2,511,628")
theo_sach("MobileNet NPU: pin ÷ số lượt (mJ)", PIN_J / LUOT["MobileNet, NPU (mobile)"][0] * 1e3, sach=2.60, nguon="2.60 mJ / 15,393,505")
for ten, (luot, *_x) in LUOT.items():
    print(f"  chiều xuôi, {ten}: pin ÷ năng lượng in ra = {PIN_J / NANG_LUONG[ten]:,.0f} lượt, sách in {luot:,}")

def co_jun(e):
    """Viết một năng lượng theo đơn vị dễ đọc."""
    for chia, don_vi in ((1e3, "kJ"), (1, "J"), (1e-3, "mJ"), (1e-6, "µJ")):
        if e >= chia:
            return f"{e / chia:,.3g} {don_vi}"
    return f"{e:.3g} J"


gpt4, kws = NANG_LUONG["GPT-4, một truy vấn (cloud)"], NANG_LUONG["phát hiện từ khoá (TinyML)"]
print(f"\nGPT-4 ÷ phát hiện từ khoá: 10^{math.log10(gpt4 / kws):.0f}")
for ten, e in NANG_LUONG.items():
    print(f"  {ten:<28} {co_jun(e):>9}  = 10^{math.log10(e / kws):.1f} lần phát hiện từ khoá")

Chiều xuôi lệch vài trăm lượt ở ba dòng giữa chỉ vì năng lượng in ra đã được làm tròn tới ba chữ số;
chiều ngược khớp cả ba. Khoảng cách từ một truy vấn GPT-4 tới một lần phát hiện từ khoá là tám bậc độ lớn,
đúng khoảng mà chương nói bảng này làm cụ thể.

Năng lượng mỗi lần chưa nói thiết bị cần bao nhiêu công suất. Công suất trung bình bằng năng lượng mỗi lần
nhân số lần mỗi giây. Chương nhắc thời lượng thật còn tuỳ nhịp suy luận, công suất lúc ngủ, cảm biến, việc
truyền và phần còn lại của thiết bị; biểu đồ dưới chỉ tính phần suy luận.

In [ ]:
NGUONG_TINY = sach(1, nguon="<1 TOPS, <1 mW") / 1e3      # W
TAN_NHIET = sach(3, nguon="~3 W", tol=0.5)              # W, tản nhiệt thụ động bền vững
nhip = np.logspace(-3, 2, 100)                          # lần mỗi giây
fig, ax = plt.subplots(figsize=(8.5, 4.6))
mau_e = [MAU[0], MAU[0], MAU[1], MAU[2], MAU[3]]
kieu = ["-", "--", "-", "-", "-"]
for (ten, e), mau, k in zip(NANG_LUONG.items(), mau_e, kieu):
    ax.loglog(nhip, e * nhip, color=mau, lw=2, ls=k)
    ax.text(nhip[-1] * 1.15, e * nhip[-1], ten, fontsize=7.5, va="center")
for muc, chu, x in ((NGUONG_TINY, f"ngưỡng TinyML: dưới {NGUONG_TINY * 1e3:.0f} mW", 1.5),
                    (TAN_NHIET, f"tản nhiệt thụ động: ~{TAN_NHIET} W", 0.3)):
    ax.axhline(muc, color="0.4", lw=0.9, ls=":")
    ax.text(x, muc * (0.42 if muc < 1 else 1.4), chu, fontsize=8, color="0.3")
ax.axvline(30, color="0.75", lw=0.8)
ax.text(30 * 0.9, 3e-8, "30 lần/giây", fontsize=7.5, color="0.4", ha="right")
ax.set_xlim(1e-3, 3e3)
ax.set_ylim(1e-8, 1e6)
ax.set_xlabel("số lần suy luận mỗi giây")
ax.set_ylabel("công suất trung bình cho suy luận (W)")
ax.set_title("Năng lượng mỗi lần × nhịp = công suất trung bình")
plt.tight_layout()
plt.show()

Ở 30 lần mỗi giây, MobileNet trên NPU cần chừng một phần mười oát, còn ResNet-50 trên Jetson cần gần nửa
oát. Phát hiện từ khoá ở nhịp đó vẫn dưới 1 mW. Đường nằm ngang 1 mW chỉ cho một mô hình cỡ phát hiện từ
khoá chạy liên tục; mọi mô hình khác phải chạy thưa hơn rất nhiều, tức phải giảm chu kỳ hoạt động
(duty cycle).

## 4. Thuế pin và tường nhiệt

Chương tách hai giới hạn. Thuế pin là bài toán ngân sách: suy luận liên tục tiêu dần năng lượng hữu hạn.
Tường nhiệt là giới hạn vật lý: điện năng rốt cuộc thành nhiệt, mà một chiếc máy mỏng chỉ tản nhiệt thụ
động qua vỏ.

Napkin Math 1.9 đặt một bộ phát hiện vật thể chạy nền, 2 W liên tục, lên một điện thoại pin 15 Wh.

**Dự đoán:** tính năng đó chạy suốt ngày được không? Nếu không, phải chạy bao nhiêu phần trăm thời gian?

In [ ]:
P_DETECT = sach(2, trich="consumes 2 W of continuous power")
PIN_WH = sach(15, trich="standard 15 Wh battery")
NGAY = sach(24, trich="expects their phone to last 24 hours")
theo_sach("thời gian chạy lý tưởng (giờ)", PIN_WH / P_DETECT, sach=7.5, trich="= 7.5 hours")
theo_sach("phần ngân sách năng lượng một ngày (%)", P_DETECT * NGAY / PIN_WH * 100, sach=320, trich="320 percent")

# thêm 1 W liên tục vào một máy vốn dùng được 24 giờ
CA_NGAY = sach(24, nguon="24 h → 9.2–11.5 h")


def con_lai(wh, them_w):
    """Thời lượng pin khi thêm một tải liên tục vào máy vốn dùng được CA_NGAY giờ."""
    return wh / (wh / CA_NGAY + them_w)


theo_sach("pin 15 Wh, thêm 1 W (giờ)", con_lai(sach(15, nguon="15 Wh–22 Wh"), 1), sach=9.2, nguon="24 h → 9.2–11.5 h")
theo_sach("pin 22 Wh, thêm 1 W (giờ)", con_lai(sach(22, nguon="15 Wh–22 Wh"), 1), sach=11.5, nguon="24 h → 9.2–11.5 h")

# cạm bẫy: nén mô hình không vượt được vật lý
theo_sach("15 Wh ở 1 W (giờ)", PIN_WH / sach(1, nguon="15 Wh ÷ 1 W = 15 h; ÷ 5 W = 3 h"), sach=15,
          nguon="15 Wh ÷ 1 W = 15 h; ÷ 5 W = 3 h")
theo_sach("15 Wh ở 5 W (giờ)", PIN_WH / sach(5, nguon="15 Wh ÷ 1 W = 15 h; ÷ 5 W = 3 h"), sach=3,
          nguon="15 Wh ÷ 1 W = 15 h; ÷ 5 W = 3 h")

In [ ]:
nen_15 = PIN_WH / NGAY       # W: điện thoại vốn dùng được 24 giờ (cùng giả định với dòng 9.2–11.5 h)
d = np.logspace(-3, 0, 200)  # chu kỳ hoạt động, 0.1 % tới 100 %
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(d * 100, PIN_WH / (P_DETECT * d), color=MAU[2], lw=2, ls="--", label="riêng tính năng, không gì khác")
ax.semilogx(d * 100, PIN_WH / (nen_15 + P_DETECT * d), color=MAU[2], lw=2, label="tính năng + phần còn lại của máy")
ax.axhline(NGAY, color="0.4", lw=0.9, ls=":")
ax.text(0.12, NGAY + 1.5, f"{NGAY} giờ", fontsize=8, color="0.3")
for phan, cho_chu in ((0.1, (0.12, 15)), (0.5, (0.9, 7))):   # giả định: mất 10 % hoặc 50 % thời lượng pin
    d_cho_phep = (PIN_WH / (NGAY * (1 - phan)) - nen_15) / P_DETECT
    ax.plot([d_cho_phep * 100], [NGAY * (1 - phan)], "o", color=MAU[2], ms=7, mec="white", mew=1.5)
    ax.annotate(f"mất {phan * 100:.0f} % thời lượng: chu kỳ {d_cho_phep * 100:.1f} %", (d_cho_phep * 100, NGAY * (1 - phan)),
                xytext=cho_chu, fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.plot([100], [PIN_WH / P_DETECT], "o", color="0.3", ms=6)
ax.annotate(f"Napkin Math 1.9, riêng tính năng: {PIN_WH / P_DETECT} giờ", (100, PIN_WH / P_DETECT), xytext=(6, 2),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(0, 60)
ax.set_xlabel("chu kỳ hoạt động của tính năng 2 W (%, thang log)")
ax.set_ylabel("thời lượng pin 15 Wh (giờ)")
ax.set_title("Thuế pin: chỉ chu kỳ hoạt động mới kéo được thời lượng về")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Chạy liên tục, tính năng 2 W cộng phần còn lại của máy kéo thời lượng xuống dưới 6 giờ. Muốn chỉ mất
một phần mười thời lượng pin (một mức sổ tay **giả định**), tính năng chỉ được chạy chưa tới 4 % thời gian.
Đó là lý do chương nói kỹ thuật nén mô hình phải giảm cả khối lượng tính toán lẫn chu kỳ hoạt động.

### Tường nhiệt

Napkin Math 1.10: một mô hình thị giác chưa tối ưu cần 12 W lúc đỉnh, trên một SoC chỉ tản được khoảng
3 W thụ động. Ở 12 W, nhiệt độ tăng chừng 1 °C mỗi giây; trong 60 s máy chạm ngưỡng 80 °C và hệ điều hành
giảm tốc do nhiệt (throttling), từ 100 FPS xuống 30 FPS. Chương cũng nhắc lý do tần số không cứu được: với
giả định minh hoạ điện áp tỉ lệ theo tần số, công suất động tăng theo lập phương tần số.

In [ ]:
P_DINH = sach(12, trich="requires 12 W peak compute")
P_BEN = sach(3, trich="approximately 3 W for passive cooling")
DOC = sach(1, trich="approximately 1 °C/s")             # °C mỗi giây ở 12 W
T_NGAT = sach(80, trich="(80 °C)")
T_GIAY = sach(60, trich="Within 60 s")
GIAM = sach(4, trich="illustrative 4× active-power reduction")
theo_sach("sau khi giảm 4 lần công suất (W)", P_DINH / GIAM, sach=3, trich="12 W ÷ 4 = 3 W")
print(f"dư địa nhiệt sau khi giảm: {P_BEN - P_DINH / GIAM:.1f} W")
theo_sach("công suất khi tăng gấp đôi tần số (lần)", 2 ** 3, sach=8, nguon="8×")
print(f"tăng 1 °C/s trong 60 s là 60 °C, nên kịch bản ngầm bắt đầu từ {T_NGAT - DOC * T_GIAY:.0f} °C")
FPS_TRUOC = sach(100, trich="The 100 FPS model suddenly drops to 30 FPS")
FPS_SAU = sach(30, trich="The 100 FPS model suddenly drops to 30 FPS")
print(f"sau giảm tốc do nhiệt: còn {FPS_SAU / FPS_TRUOC * 100:.0f} % thông lượng; tường công suất của chương: "
      f"{sach(60, nguon='60 → 15 FPS')} → {sach(15, nguon='60 → 15 FPS')} FPS sau 1 phút")

Để thấy vì sao 3 W là "vừa đủ" chứ không phải "có dư", sổ tay dùng một mô hình nhiệt hai tham số, là
**giả định** của sổ tay chứ không phải của chương: nhiệt dung $C$ và nhiệt trở $R$ ra môi trường,

$$C\,\frac{dT}{dt} = P - \frac{T - T_\text{môi trường}}{R}.$$

Hai tham số được chọn để khớp đúng hai điều chương nói: độ dốc ban đầu 1 °C/s ở 12 W cho $C$, và việc
3 W là mức bền vững ngay tại ngưỡng 80 °C cho $R$, với môi trường 20 °C như kịch bản ngầm định. Ngoài hai
điều đó, mô hình không được kiểm với gì cả.

In [ ]:
T_MT = T_NGAT - DOC * T_GIAY              # 20 °C, suy ra từ kịch bản
C_NHIET = P_DINH / DOC                    # J mỗi °C
R_NHIET = (T_NGAT - T_MT) / P_BEN         # °C mỗi W
tau = R_NHIET * C_NHIET
t = np.linspace(0, 1200, 1201)


def nhiet_do(P, t):
    """Lời giải đóng của mô hình hai tham số, bắt đầu từ nhiệt độ môi trường."""
    return T_MT + P * R_NHIET * (1 - np.exp(-t / tau))


TRAN = 145   # °C: trên mức này đường bị cắt khỏi biểu đồ


def cat(y):
    """Bỏ phần đường vượt khỏi khung, thay vì ép nó nằm ngang ở mép trên."""
    return np.where(y > TRAN, np.nan, y)


fig, ax = plt.subplots(figsize=(8, 4))
for P, mau in ((P_DINH, MAU[1]), (6, MAU[3]), (P_BEN, MAU[2])):
    ax.plot(t, cat(nhiet_do(P, t)), color=mau, lw=2)
ax.text(185, 138, f"{P_DINH:g} W", fontsize=8)
ax.text(1210, nhiet_do(6, 1200), "6 W", fontsize=8, va="center")
ax.text(1210, nhiet_do(P_BEN, 1200) - 4, f"{P_BEN:g} W", fontsize=8, va="center")
ax.plot(t, cat(T_MT + DOC * t), color="0.6", lw=1, ls="--")
ax.annotate("đường thẳng 1 °C/s của chương", (110, T_MT + DOC * 110), xytext=(220, 133), fontsize=8, color="0.4",
            arrowprops=dict(arrowstyle="-", color="0.6", lw=0.8))
ax.axhline(T_NGAT, color="0.3", lw=0.9, ls=":")
ax.text(620, T_NGAT + 2.5, f"ngưỡng giảm tốc do nhiệt: {T_NGAT} °C", fontsize=8, color="0.3")
t12 = -tau * math.log(1 - (T_NGAT - T_MT) / (P_DINH * R_NHIET))
ax.plot([t12], [T_NGAT], "o", color=MAU[1], ms=7, mec="white", mew=1.5)
ax.annotate(f"12 W chạm ngưỡng sau {t12:.0f} s", (t12, T_NGAT), xytext=(300, 40), fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlim(0, 1300)
ax.set_ylim(15, TRAN)
ax.set_xlabel("thời gian (s)")
ax.set_ylabel("nhiệt độ (°C)")
ax.set_title("Mô hình nhiệt giả định: 3 W tiến sát 80 °C nhưng không vượt")
plt.tight_layout()
plt.show()
print(f"hằng số thời gian RC = {tau:.0f} s; ở 3 W, sau 20 phút máy ở {nhiet_do(P_BEN, 1200):.1f} °C và chỉ tiệm cận {T_NGAT} °C")

Mô hình chạm ngưỡng chậm hơn đường thẳng của chương chừng mười giây, vì máy càng nóng thì càng tản được
nhiều.
Điểm đáng giữ lại là đường 3 W: nó tiến tới đúng 80 °C mà không bao giờ vượt, nên bất kỳ phần công suất
nào thêm vào, như thêm một tác vụ nền, đều đẩy máy qua ngưỡng. Đó là ý của chương khi nói giảm 4 lần công
suất không tạo ra dư địa nhiệt, chỉ vừa chạm giới hạn bền vững. Pin lớn hơn cũng không đổi được đường này.

### Kho năng lượng của một cảm biến TinyML

Chương nêu một cạm bẫy của thiết bị thu hoạch năng lượng: thiết bị có thể đạt ngân sách công suất trung
bình dài hạn mà vẫn thất bại, nếu năng lượng tích trữ cạn trước lần thu hoạch kế tiếp. Ô dưới mô phỏng
một cảm biến phát hiện từ khoá. Chỉ năng lượng mỗi lần suy luận là của chương; nhịp suy luận, công suất
ngủ, pin mặt trời và kho đều là **giả định** của sổ tay.

In [ ]:
E_KWS = NANG_LUONG["phát hiện từ khoá (TinyML)"]
NHIP_KWS = 10            # giả định: 10 lần mỗi giây
P_NGU = 20e-6            # giả định: 20 µW lúc ngủ
P_THU = 300e-6           # giả định: 300 µW từ pin mặt trời, chỉ 12 giờ ban ngày
p_tieu = E_KWS * NHIP_KWS + P_NGU
print(f"tiêu thụ trung bình: {p_tieu * 1e6:.0f} µW; thu trung bình cả ngày: {P_THU / 2 * 1e6:.0f} µW "
      f"→ ngân sách trung bình {'ĐẠT' if P_THU / 2 >= p_tieu else 'TRƯỢT'}")

gio = np.arange(0, 72 * 3600, 60) / 3600   # 72 giờ, bước 1 phút
co_nang = ((gio % 24) >= 6) & ((gio % 24) < 18)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
for kho, mau in ((3.0, MAU[1]), (8.0, MAU[0])):          # giả định: kho 3 J và 8 J, đầy lúc bắt đầu
    e, muc, tat = kho, [], 0
    for sang in co_nang:
        e = min(kho, e + (P_THU * sang - p_tieu) * 60)
        if e <= 0:
            e, tat = 0.0, tat + 1
        muc.append(e)
    ax.plot(gio, muc, color=mau, lw=2, label=f"kho {kho:g} J: mất {tat / 60:.1f} giờ mỗi 72 giờ")
for ngay in range(3):
    ax.axvspan(ngay * 24 + 6, ngay * 24 + 18, color=MAU[3], alpha=0.12, lw=0)
ax.text(12, 8.6, "ban ngày", ha="center", fontsize=8, color="0.35")
ax.set_xlim(0, 72)
ax.set_ylim(0, 11)
ax.set_xticks(range(0, 73, 12))
ax.set_xlabel("giờ")
ax.set_ylabel("năng lượng trong kho (J)")
ax.set_title("Cùng công suất trung bình, kho nhỏ cạn trước bình minh")
ax.legend(loc="upper center", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
print(f"một đêm 12 giờ cần {p_tieu * 12 * 3600:.2f} J trong kho")

Hai cảm biến có cùng ngân sách trung bình, và cả hai đều đạt. Chỉ cảm biến có kho đủ lớn cho một đêm mới
chạy liên tục; cảm biến kia tắt mỗi đêm trước bình minh. Vì vậy chương nói chu kỳ hoạt động phải tính tới
cả năng lượng cho mỗi lần suy luận, lẫn thời điểm năng lượng thực sự có.

## 5. Tiền: tổng chi phí sở hữu

Napkin Math 1.4 xét một hệ thống thị giác phục vụ 1 triệu lần suy luận mỗi ngày, mỗi phản hồi 100 KB,
trong 3 năm. Bên cloud là 4 máy GPU cố định, bên edge là một máy chủ tự vận hành.

**Dự đoán:** khoản nào lớn nhất ở bên edge, phần cứng hay thứ khác?

In [ ]:
# Bên cloud
GPU = sach(4, trich=r"4 instances \(\times\) 8760 h/year \(\times\) $0.75/hr")
GIO_NAM = sach(8760, trich=r"4 instances \(\times\) 8760 h/year \(\times\) $0.75/hr")
GIA_GPU = sach(0.75, trich=r"4 instances \(\times\) 8760 h/year \(\times\) $0.75/hr")
LUOT_NGAY = sach(1e6, trich="1M daily inferences")
KB_PHAN_HOI = sach(100, trich="10 ms latency, 100 KB response")
theo_sach("dữ liệu ra mỗi ngày (GB)", LUOT_NGAY * KB_PHAN_HOI * KB / GB, sach=100,
          trich=r"100 GB/day \(\times\) 365 d/year \(\times\) $0.09/GB")
NGAY_NAM = sach(365, trich=r"100 GB/day \(\times\) 365 d/year \(\times\) $0.09/GB")
GIA_RA = sach(0.09, trich=r"100 GB/day \(\times\) 365 d/year \(\times\) $0.09/GB")
cloud = {
    "GPU suy luận": GPU * GIO_NAM * GIA_GPU,
    "dữ liệu ra": LUOT_NGAY * KB_PHAN_HOI * KB / GB * NGAY_NAM * GIA_RA,
    "cân bằng tải": sach(3723, nguon="$26,280 / $3,285 / $3,723 / $2,000"),
    "theo dõi, ghi log": sach(2000, nguon="$26,280 / $3,285 / $3,723 / $2,000"),
}
theo_sach("cloud: GPU suy luận ($/năm)", cloud["GPU suy luận"], sach=26_280, nguon="$26,280 / $3,285 / $3,723 / $2,000")
theo_sach("cloud: dữ liệu ra ($/năm)", cloud["dữ liệu ra"], sach=3285, nguon="$26,280 / $3,285 / $3,723 / $2,000")
theo_sach("cloud: tổng ($/năm)", sum(cloud.values()), sach=35_288, nguon="$35,288 / năm")

# Bên edge
CAPEX = sach(15_000, nguon="$15,000 ÷ 3 năm, 300 W, $0.12/kWh, 0.1 FTE")
NAM = sach(3, nguon="$15,000 ÷ 3 năm, 300 W, $0.12/kWh, 0.1 FTE")
W_MAY = sach(300, nguon="$15,000 ÷ 3 năm, 300 W, $0.12/kWh, 0.1 FTE")
GIA_DIEN = sach(0.12, nguon="$15,000 ÷ 3 năm, 300 W, $0.12/kWh, 0.1 FTE")
FTE = sach(0.1, nguon="$15,000 ÷ 3 năm, 300 W, $0.12/kWh, 0.1 FTE")
LUONG = sach(150_000, trich=r"0.1 FTE \(\times\) $150,000 salary")
LAM_MAT = sach(30, nguon="~$315, ~30 %, ~$1,200", tol=0.5) / 100
edge = {
    "phần cứng (khấu hao)": CAPEX / NAM,
    "điện 24/7": W_MAY / 1000 * GIO_NAM * GIA_DIEN,
    "làm mát": LAM_MAT * W_MAY / 1000 * GIO_NAM * GIA_DIEN,
    "đường truyền": sach(1200, nguon="~$315, ~30 %, ~$1,200", tol=0.5),
    "nhân công DevOps": FTE * LUONG,
}
theo_sach("edge: điện ($/năm)", edge["điện 24/7"], sach=315.4, trich="~$315.4", tol=0.05)
theo_sach("edge: làm mát ($/năm)", edge["làm mát"], sach=94.6, trich="~$94.6", tol=0.05)
theo_sach("edge: tổng ($/năm)", sum(edge.values()), sach=21_610, nguon="$21,610 / năm")
khoang = sum(cloud.values()) - sum(edge.values())
theo_sach("chênh lệch mỗi năm ($)", khoang, sach=13_678, trich="~$13,678/year", tol=0.5)
theo_sach("edge rẻ hơn (%)", khoang / sum(cloud.values()) * 100, sach=38.8, trich="38.8 percent less")
theo_sach("nhân công trong TCO edge (%)", edge["nhân công DevOps"] / sum(edge.values()) * 100, sach=69.4,
          trich="≈ 69.4 percent", tol=0.05)
co_dinh = sum(v for k, v in cloud.items() if k != "dữ liệu ra")
assert co_dinh > sum(edge.values())   # phần cố định của cloud đã đắt hơn cả bên edge
print(f"phần cloud không đổi theo lưu lượng: {co_dinh:,.0f} $/năm, đã hơn tổng edge {sum(edge.values()):,.0f} $/năm")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
nhan = [f"cloud · {k}" for k in cloud] + [f"edge · {k}" for k in edge]
gia_tri = list(cloud.values()) + list(edge.values())
ax.barh(nhan, gia_tri, color=[MAU[0]] * len(cloud) + [MAU[1]] * len(edge), height=0.65)
for i, v in enumerate(gia_tri):
    ax.text(v + 300, i, f"{v:,.0f}", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlim(0, 31_000)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xlabel("đô la mỗi năm")
ax.set_title(f"TCO mỗi năm: cloud {sum(cloud.values()):,.0f} đô la, edge {sum(edge.values()):,.0f} đô la")
plt.tight_layout()
plt.show()

Khoản lớn nhất bên edge là nhân công, không phải phần cứng. Vì chi phí bên cloud ở đây gần như cố định,
chương lưu ý không thể lấy tỉ số hai tổng làm điểm hoà vốn theo lưu lượng. Một biến có thể quét được là
nhân công: edge còn rẻ hơn tới bao nhiêu FTE (nhân sự toàn thời gian)?

In [ ]:
ngoai_nhan_cong = sum(v for k, v in edge.items() if k != "nhân công DevOps")
fte_hoa_von = (sum(cloud.values()) - ngoai_nhan_cong) / LUONG
print(f"điểm hoà vốn: {fte_hoa_von:.3f} FTE (sách dùng {FTE} FTE); trên mức đó edge đắt hơn cloud")

# cạm bẫy "ít tài nguyên thì ít tiền", theo tháng
CLOUD_THANG = sach(2000, nguon="$2,000 vs $6,000 / tháng, 3×")
edge_thang = (sach(500, trich="$500/month in edge-hardware amortization") + sach(3000, nguon="$3,000 + $500 + $2,000")
              + sach(500, nguon="$3,000 + $500 + $2,000") + sach(2000, nguon="$3,000 + $500 + $2,000"))
theo_sach("edge mỗi tháng, đủ các khoản ($)", edge_thang, sach=6000, nguon="$2,000 vs $6,000 / tháng, 3×")
theo_sach("edge so với cloud (lần)", edge_thang / CLOUD_THANG, sach=3, nguon="$2,000 vs $6,000 / tháng, 3×")

# vốn đầu tư ban đầu cho 1,000 điểm edge
DIEM = sach(1000, nguon="$500–2,000 mỗi điểm, 1,000 điểm → $500,000–2,000,000")
theo_sach("1,000 điểm × 500 $ ($)", DIEM * sach(500, nguon="$500–2,000 mỗi điểm, 1,000 điểm → $500,000–2,000,000"),
          sach=500_000, nguon="$500–2,000 mỗi điểm, 1,000 điểm → $500,000–2,000,000")
theo_sach("1,000 điểm × 2,000 $ ($)", DIEM * sach(2000, nguon="$500–2,000 mỗi điểm, 1,000 điểm → $500,000–2,000,000"),
          sach=2_000_000, nguon="$500–2,000 mỗi điểm, 1,000 điểm → $500,000–2,000,000")

### Chọn bộ tăng tốc cho 500 cửa hàng

Napkin Math 1.8: một chuỗi bán lẻ chạy phát hiện người ở 500 cửa hàng, mỗi cửa hàng 20 camera ở 15 FPS,
với YOLOv8-nano tốn 8.7 GFLOP mỗi lần và dư địa gấp 2. Chương dặn tính từ mức một bo mạch duy trì được,
không phải mức đỉnh trên bảng thông số.

In [ ]:
CAM = sach(20, nguon="20 camera × 15 FPS = 300 suy luận/s")
FPS = sach(15, nguon="20 camera × 15 FPS = 300 suy luận/s")
YOLO = sach(8.7, nguon="8.7 GFLOP, 2610 GFLOP/s, ~5.22 TOPS", tol=0.05)
DU = sach(2, trich=r"2610 GFLOP/s \(\times\) 2 (headroom)")
CUA_HANG = sach(500, nguon="500 cửa hàng")
theo_sach("suy luận mỗi giây một cửa hàng", CAM * FPS, sach=300, nguon="20 camera × 15 FPS = 300 suy luận/s")
theo_sach("tính toán một cửa hàng (GFLOP/s)", CAM * FPS * YOLO, sach=2610, nguon="8.7 GFLOP, 2610 GFLOP/s, ~5.22 TOPS", tol=0.5)
can = CAM * FPS * YOLO * DU / 1000
theo_sach("thông lượng cần, có dư địa (TOPS)", can, sach=5.22, nguon="8.7 GFLOP, 2610 GFLOP/s, ~5.22 TOPS", tol=0.005)

CORAL_DINH = sach(4, trich="rated at 4 TOPS peak")
CORAL_THAT = sach(2, trich="about 2 TOPS after 50 percent derating")
theo_sach("bo Coral cần, chưa làm tròn", can / CORAL_THAT, sach=2.6, trich="= 2.6 boards")
so_bo = math.ceil(can / CORAL_THAT)
theo_sach("bo Coral mỗi cửa hàng", so_bo, sach=3, trich="3 boards/store")
theo_sach("3 bo Coral, mức đỉnh (TOPS)", so_bo * CORAL_DINH, sach=12, nguon="12 → 6 TOPS / $450 / $225,000")
theo_sach("3 bo Coral, mức duy trì (TOPS)", so_bo * CORAL_THAT, sach=6, nguon="12 → 6 TOPS / $450 / $225,000")

LUA_CHON = {   # TOPS dùng được mỗi cửa hàng, giá phần cứng mỗi cửa hàng
    "Jetson Orin NX": (sach(100, nguon="100 TOPS / $600 / $300,000"), sach(600, nguon="100 TOPS / $600 / $300,000")),
    "Intel NUC + Movidius": (sach(1, nguon="1 TOPS / $400 / $200,000"), sach(400, nguon="1 TOPS / $400 / $200,000")),
    "3 bo Google Coral": (so_bo * CORAL_THAT, sach(450, nguon="12 → 6 TOPS / $450 / $225,000")),
}
theo_sach("Jetson Orin NX, cả chuỗi ($)", CUA_HANG * LUA_CHON["Jetson Orin NX"][1], sach=300_000,
          nguon="100 TOPS / $600 / $300,000")
theo_sach("Intel NUC + Movidius, cả chuỗi ($)", CUA_HANG * LUA_CHON["Intel NUC + Movidius"][1], sach=200_000,
          nguon="1 TOPS / $400 / $200,000")
theo_sach("3 bo Google Coral, cả chuỗi ($)", CUA_HANG * LUA_CHON["3 bo Google Coral"][1], sach=225_000,
          nguon="12 → 6 TOPS / $450 / $225,000")
for ten, (tops, gia) in LUA_CHON.items():
    print(f"   {ten:<22} {tops:>5g} TOPS so với {can:.2f} TOPS cần → {'đủ' if tops >= can else 'KHÔNG đủ'}, "
          f"{CUA_HANG * gia:,} $ cho cả chuỗi")

KW = sach(0.006, trich="0.006 kW")
dien = KW * CUA_HANG * GIO_NAM * NAM * GIA_DIEN
theo_sach("điện Coral, 3 năm ($)", dien, sach=9460.8, trich="$9,460.8")
theo_sach("Coral, tổng 3 năm ($)", CUA_HANG * LUA_CHON["3 bo Google Coral"][1] + dien, sach=234_460.8,
          trich="$234,460.8 total")
theo_sach("vốn mỗi cửa hàng, Jetson ÷ Coral", LUA_CHON["Jetson Orin NX"][1] / LUA_CHON["3 bo Google Coral"][1],
          sach=1.3, trich="about 1.3× lower per-store hardware CapEx")
# chương không ghi mốc cloud được tính thế nào; nó khớp đúng với một GPU mỗi cửa hàng ở giá A10G của bảng TCO cloud
theo_sach("mốc cloud: một GPU mỗi cửa hàng, 3 năm ($)", CUA_HANG * GIO_NAM * NAM * GIA_GPU, sach=9_855_000,
          trich="baseline at ~$9,855,000. Result", tol=0.5)

Intel NUC rẻ nhất nhưng không đủ thông lượng (throughput), và chương không coi nó là lựa chọn. Ba bo Coral là phương án
edge rẻ nhất vẫn đủ thông lượng, còn Jetson là phương án một thiết bị, đơn giản hơn khi độ phức tạp tích
hợp quan trọng hơn giá phần cứng. Chương kết luận đây là bài toán năng lực và chi phí, không phải bài toán
tên thiết bị. Mốc so sánh cloud của chương lớn hơn tổng của Coral hơn bốn mươi lần. Nhưng tổng đó chưa
tính nhân công vận hành một đội máy edge 500 điểm, khoản mà Napkin Math 1.4 cho thấy là lớn nhất, và chương
dặn phải cân khoản tiết kiệm phần cứng với gánh nặng điều phối và bảo trì một đội máy chia nhỏ như vậy.

## 6. Cloud: cái giá của GPT-3

Chương mở phần cloud bằng phép tính huấn luyện GPT-3: 3,634.3 PFLOP-day, trên 10,000 GPU V100 chạy khoảng
15 ngày. Một PFLOP-day là $10^{15}$ phép tính mỗi giây trong một ngày.

**Dự đoán:** một thiết bị mỗi môi trường, chạy ở đúng thông lượng đỉnh, cần bao lâu?

In [ ]:
PF_NGAY = sach(3634.3, nguon="3,634.3 PFLOP-days")
V100 = sach(10_000, nguon="10,000 V100, ~15 ngày", tol=0.5)
NGAY_HL = sach(15, nguon="10,000 V100, ~15 ngày", tol=0.5)
tong_flop = PF_NGAY * 1e15 * 86_400
print(f"tổng: {tong_flop:.3g} phép tính; ngầm định mỗi V100 duy trì {PF_NGAY * 1e15 / (V100 * NGAY_HL) / 1e12:.1f} TFLOP/s")

DINH = {   # phép tính mỗi giây ở thông lượng đỉnh; mỗi con số ở một độ chính xác khác nhau
    "TPU v4 Pod (BF16)": sach(1.1, trich="1.1 EFLOP/s BF16") * 1e18,
    "DGX Spark (FP4 thưa)": sach(1, trich="1 PFLOP/s sparse FP4") * 1e15,
    "NPU điện thoại (INT8)": sach(35, trich="~35 TOPS (INT8)", tol=0.5) * 1e12,
    "TinyML (dưới 1 TOPS)": sach(1, trich="< 1 TOPS") * 1e12,
}
for ten, r in DINH.items():
    ngay = tong_flop / r / 86_400
    print(f"{ten:<24} {ngay:>12,.1f} ngày" + (f" = {ngay / 365:>7,.0f} năm" if ngay >= 365 else ""))
print("(đều là cận dưới: thông lượng đỉnh, hiệu suất 100 %, và TinyML còn dưới 1 TOPS)")

Ngay cả với thông lượng đỉnh, hiệu suất 100 % và những độ chính xác không ai dùng để huấn luyện (FP4 thưa,
INT8), một DGX Spark cần khoảng mười năm và một NPU điện thoại cần vài trăm năm. Chỉ một cụm cỡ trung tâm
dữ liệu mới làm xong trong vài ngày. Đó là mệnh đề định nghĩa của cloud ML trong chương: khi chịu được độ
trễ, cloud cho quy mô tính toán mà không môi trường nào khác sánh được.

## Thử thêm

1. Ở phần 2, thêm `"INT4": 0.5` vào `BYTE` và vẽ lại bảng với INT4. Ô nào đổi từ "thiếu" sang "vừa"?
   Llama 70B có xuống được điện thoại không?
2. Ở phần 4, đổi `P_THU` trong ô kho năng lượng thành 250 µW, rồi 200 µW. Ngân sách trung bình còn đạt
   không, và kho 8 J còn qua được đêm không?
3. Ở phần 5, đổi `FTE` thành 0.2 rồi chạy lại ô TCO. Bên nào rẻ hơn, và kết quả có khớp điểm hoà vốn
   bạn đã tính không? (Các dấu ✗ khi đổi tham số là điều được chờ đợi.)

## Tóm lại

* Bốn môi trường cách nhau theo bậc độ lớn: chín bậc công suất từ MW tới mW, và 250 triệu lần bộ nhớ từ
  TPU v4 Pod tới ESP32-S3; chỉ thang log mới cho thấy cả bốn.
* Lượng tử hóa dời ranh giới bộ nhớ 4 lần, còn từ vi điều khiển lên đích edge cấp thấp đã là một nghìn
  lần; khi phải gói trong 64 KB thì phải đổi kiến trúc, và qua cấp bộ nhớ chưa phải qua cấp thời gian.
* Năng lượng mỗi lần suy luận trải tám bậc, từ một truy vấn GPT-4 tới một lần phát hiện từ khoá; nhân với
  nhịp suy luận mới ra công suất mà pin và lớp vỏ phải chịu.
* Thuế pin là ngân sách, giải được bằng chu kỳ hoạt động; tường nhiệt là trần vật lý, và mức 3 W chỉ vừa
  chạm nó; với thu hoạch năng lượng, kho phải đủ cho lúc không có năng lượng.
* TCO edge chủ yếu là nhân công, nên con số nhân sự quyết định bên nào rẻ hơn; chọn phần cứng edge là bài
  toán năng lực và chi phí, không phải bài toán tên thiết bị.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Hệ thống học máy, bốn môi trường triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch02-ml-systems/), dựng từ chương
[*ML Systems*](https://mlsysbook.ai/vol1/ml_systems/ml_systems.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_02_ml_systems.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.